In [0]:
%pip install catboost openpyxl

In [0]:
from pathlib import Path
import re
import sys
import time

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

SEED = 42
N_FOLDS = 5
MIN_PRODUKTE_BIG1 = 5
MIN_GRUPPE_PRODUKTE = 6            # Gruppen mit weniger Produkten werden übersprungen
HAUPTGRUPPE_ZEICHEN = 6            # Präfix-Länge für die Hauptgruppe
K_NB_B3 = 10
K_NACHBARN_BIG2 = 10
TOP_PRODUKTE = 120
MAX_LISTENZEILEN = 200
FEHLT_WERT = "OHNE_ANGABE"

BOM_DATEI = "191007_stuekliste_neu.xlsx"

basis = Path("/Workspace/Users/khalil-said.albert@de.abb.com/Daten")
home = Path("/Workspace/Users/khalil-said.albert@de.abb.com")
LAUF_NAME = Path(BOM_DATEI).stem

# Datei kann in basis, home oder home/Stueklisten liegen
_suchpfade = [basis / BOM_DATEI, home / BOM_DATEI, home / "Stueklisten" / BOM_DATEI]
bom_pfad = next((p for p in _suchpfade if p.is_file()), None)
assert bom_pfad is not None, f"BOM-Datei nicht gefunden: {BOM_DATEI} (gesucht in {[str(p.parent) for p in _suchpfade]})"
code_dir = home / "Code"
assert (code_dir / "bom_core_new_v4.py").is_file(), "bom_core_new_v4.py fehlt in Code/"
assert (code_dir / "layer2_profile.py").is_file() or (basis / "layer2_profile.py").is_file(), "layer2_profile.py fehlt"

for p in [str(code_dir), str(basis), str(home)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import bom_core_new_v4 as bc
import layer2_profile as L2

L2.HAUPTGRUPPE_ZEICHEN = HAUPTGRUPPE_ZEICHEN

start = time.perf_counter()
sauber, protokoll = bc.clean_bom(str(bom_pfad))

roh = pd.DataFrame(sauber).copy()

# Cleaner-Protokoll
warns = protokoll.get("warnungen", [])
if warns:
    print(f"Cleaner-Warnungen ({len(warns)}):")
    for w in warns:
        print(f"  - {w}")
spalten_info = protokoll.get("spalten", [])
if spalten_info:
    print(f"\nErkannte Spalten ({len(spalten_info)}):")
    for sp in spalten_info:
        print(f"  {sp}")

HG_UEBERSPRUNGEN = {}  # {gruppe: grund} – wird in Cmd 3 gefüllt

display(pd.DataFrame([{
    "Datei": bom_pfad.name,
    "Lauf": LAUF_NAME,
    "Positionszeilen": len(roh),
    "Produkte": roh["erzeugnis"].astype(str).nunique(),
    "Produktblöcke": len(roh[["erzeugnis", "block"]].drop_duplicates()),
    "Laufzeit in Sekunden": round(time.perf_counter() - start, 1),
}]))

In [0]:
def haeufigster_text(werte):
    s = werte.fillna("").astype(str).str.strip()
    s = s.loc[s.ne("")]
    return s.value_counts().index[0] if not s.empty else ""


def spannung_text(text):
    t = str(text).replace("*", " ").replace("+", " ").replace("_", " ")
    treffer = re.findall(r"(\d+(?:[-/]\d+)?)\s*V\s*(DC|AC)?", t, re.I)
    werte = set()
    for wert, art in treffer:
        art = art.upper() or (
            "AC" if re.search(r"\d+(?:-\d+)?\s*HZ", t, re.I) else "UNBEKANNT"
        )
        werte.add(f"{wert}_{art}")
    if len(werte) == 1:
        return next(iter(werte))
    if werte:
        return "MEHRDEUTIG"
    return FEHLT_WERT


def eindeutig(werte):
    werte = set(pd.Series(werte).fillna(FEHLT_WERT).astype(str))
    return next(iter(werte)) if len(werte) == 1 else "MEHRDEUTIG"


def menge_als_zahl(wert):
    if pd.isna(wert):
        return np.nan
    if isinstance(wert, (int, float, np.integer, np.floating)):
        return float(wert)
    text = str(wert).strip().replace(" ", "")
    if "," in text and "." not in text:
        text = text.replace(",", ".")
    return pd.to_numeric(text, errors="coerce")


q = roh.copy()
for c in ["erzeugnis", "block", "komponente"]:
    q[c] = q[c].astype(str)
q["hauptgruppe"] = q["erzeugnis"].map(L2.hauptgruppe_von)

# Gruppenübersicht ableiten
_gruppen = (
    q.groupby("hauptgruppe")
    .agg(
        Produkte=("erzeugnis", "nunique"),
        Blöcke=("block", "nunique"),
        Komponenten=("komponente", "nunique"),
        Zeilen=("erzeugnis", "size"),
    )
    .reset_index()
    .sort_values("Produkte", ascending=False)
)
_gruppen["Status"] = np.where(
    _gruppen["Produkte"].ge(MIN_GRUPPE_PRODUKTE),
    "wird verarbeitet",
    "\u00fcbersprungen: < " + str(MIN_GRUPPE_PRODUKTE) + " Produkte",
)
print("Gruppen\u00fcbersicht:")
display(_gruppen.rename(columns={"hauptgruppe": "Hauptgruppe"}))

for _, g in _gruppen.loc[_gruppen["Produkte"].lt(MIN_GRUPPE_PRODUKTE)].iterrows():
    HG_UEBERSPRUNGEN[g["hauptgruppe"]] = f"< {MIN_GRUPPE_PRODUKTE} Produkte ({int(g['Produkte'])})"

HAUPTGRUPPEN = tuple(_gruppen.loc[_gruppen["Produkte"].ge(MIN_GRUPPE_PRODUKTE), "hauptgruppe"].tolist())
print(f"\nVerarbeitet: {', '.join(HAUPTGRUPPEN)} ({len(HAUPTGRUPPEN)} Gruppen)")
if HG_UEBERSPRUNGEN:
    print(f"\u00dcbersprungen: {', '.join(f'{k} ({v})' for k, v in HG_UEBERSPRUNGEN.items())}")

q = q.loc[q["hauptgruppe"].isin(HAUPTGRUPPEN)].copy()
q["menge_num"] = q["menge"].map(menge_als_zahl)
q["tiefe_num"] = pd.to_numeric(q["tiefe"], errors="coerce")
q["stufe_num"] = pd.to_numeric(q["stufe"], errors="coerce")

produktnamen = q.groupby("erzeugnis")["erzeugnis_txt"].agg(haeufigster_text)
komponentennamen = q.groupby("komponente")["komponente_txt"].agg(haeufigster_text)

positionen = q[["erzeugnis", "block", "komponente"]].copy()
assert positionen.notna().all().all(), "Produkt, Block oder Komponente fehlt"

block_sets = (
    positionen.groupby(["erzeugnis", "block"], sort=False)["komponente"]
    .agg(lambda werte: tuple(sorted(set(werte))))
    .reset_index(name="komponentenset")
)

fassungen = (
    block_sets[["erzeugnis", "komponentenset"]]
    .drop_duplicates(["erzeugnis", "komponentenset"])
    .reset_index(drop=True)
)
fassungen["fassung_id"] = np.arange(len(fassungen))
fassungen["produktgewicht"] = 1.0 / fassungen.groupby("erzeugnis")["fassung_id"].transform("count")
fassungen["hauptgruppe"] = fassungen["erzeugnis"].map(L2.hauptgruppe_von)

block_fassungen = block_sets.merge(
    fassungen[["erzeugnis", "komponentenset", "fassung_id", "produktgewicht", "hauptgruppe"]],
    on=["erzeugnis", "komponentenset"],
    how="left",
    validate="many_to_one",
)
block_fassungen["bkey"] = block_fassungen["erzeugnis"] + "|" + block_fassungen["block"]

X_je_hauptgruppe = {}
info_je_hauptgruppe = {}
fold_je_hauptgruppe = {}
block_sets_je_hauptgruppe = {}
block_fassungen_je_hauptgruppe = {}
rep_block_je_fassung_je_hg = {}
rep_bkey_je_fassung_je_hg = {}

for hg in HAUPTGRUPPEN:
    f_hg = fassungen.loc[fassungen["hauptgruppe"].eq(hg)].copy()
    vorhanden = (
        f_hg[["fassung_id", "komponentenset"]]
        .explode("komponentenset")
        .rename(columns={"komponentenset": "komponente"})
    )
    X = pd.crosstab(vorhanden["fassung_id"], vorhanden["komponente"])
    X = X.reindex(index=f_hg["fassung_id"], fill_value=0)
    X = (X > 0).astype("int8")

    produkte = np.array(sorted(f_hg["erzeugnis"].unique()))
    np.random.default_rng(SEED + sum(map(ord, hg))).shuffle(produkte)
    fold_map = {produkt: i % N_FOLDS for i, produkt in enumerate(produkte)}

    info = f_hg.set_index("fassung_id").loc[X.index].copy()
    info["fold"] = info["erzeugnis"].map(fold_map)

    bf = block_fassungen.loc[block_fassungen["hauptgruppe"].eq(hg)].copy()
    bf = bf.sort_values(["erzeugnis", "block"])

    X_je_hauptgruppe[hg] = X
    info_je_hauptgruppe[hg] = info
    fold_je_hauptgruppe[hg] = fold_map
    block_sets_je_hauptgruppe[hg] = block_sets.loc[
        block_sets["erzeugnis"].map(L2.hauptgruppe_von) == hg
    ].copy()
    block_fassungen_je_hauptgruppe[hg] = bf
    rep_block_je_fassung_je_hg[hg] = bf.groupby("fassung_id")["block"].first()
    rep_bkey_je_fassung_je_hg[hg] = bf.groupby("fassung_id")["bkey"].first()

summary_shared = []
for hg in HAUPTGRUPPEN:
    summary_shared.append({
        "Hauptgruppe": hg,
        "Produkte": info_je_hauptgruppe[hg]["erzeugnis"].nunique(),
        "Fassungen": X_je_hauptgruppe[hg].shape[0],
        "Blöcke": block_fassungen_je_hauptgruppe[hg]["bkey"].nunique(),
        "Komponenten": X_je_hauptgruppe[hg].shape[1],
    })

display(pd.DataFrame(summary_shared))

In [0]:
BIG1_KATEGORIAL = [
    "pruefkomponente",
    "kleinere_gruppe",
    "produkt_spannung",
    "komponente_spannung",
    "komponente_basis",
]

paare_je_hauptgruppe = {}
y_je_hauptgruppe = {}
features_big1_je_hauptgruppe = {}
modelle_big1_je_hauptgruppe = {}
spalten_big1_je_hauptgruppe = {}
ergebnisse_big1_je_hauptgruppe = {}
auswertung_big1 = []

for hg in HAUPTGRUPPEN:
    X = X_je_hauptgruppe[hg]
    info = info_je_hauptgruppe[hg]

    paare = (
        X.rename_axis(index="fassung_id", columns="komponente")
        .stack()
        .rename("vorhanden")
        .reset_index()
    )
    meta = info[["erzeugnis", "produktgewicht", "fold"]].rename_axis("fassung_id").reset_index()
    paare = paare.merge(meta, on="fassung_id", how="left", validate="many_to_one")

    kontext = X.loc[paare["fassung_id"]].to_numpy(copy=True)
    ziel_spalten = X.columns.get_indexer(paare["komponente"])
    y = paare["vorhanden"].to_numpy(copy=True)
    zeilen = np.arange(len(paare))
    assert np.array_equal(kontext[zeilen, ziel_spalten], y)
    kontext[zeilen, ziel_spalten] = 0

    features = pd.DataFrame(
        kontext,
        columns=[f"kontext_{i}" for i in range(X.shape[1])],
        index=paare.index,
    )
    features["pruefkomponente"] = paare["komponente"].astype(str)

    quelle = q.loc[q["hauptgruppe"].eq(hg), [
        "erzeugnis", "erzeugnis_txt", "komponente", "komponente_txt"
    ]].copy()
    quelle["kleinere_gruppe"] = quelle["erzeugnis_txt"].fillna("").map(
        lambda t: L2.typ_von(str(t)) or FEHLT_WERT
    )
    quelle["produkt_spannung"] = quelle["erzeugnis_txt"].map(spannung_text)
    quelle["komponente_spannung"] = quelle["komponente_txt"].map(spannung_text)

    produkt_merkmale = quelle.groupby("erzeugnis")[["kleinere_gruppe", "produkt_spannung"]].agg(eindeutig)
    komponenten_merkmale = quelle.groupby("komponente")[["komponente_spannung"]].agg(eindeutig)
    komponenten_merkmale["komponente_basis"] = (
        komponenten_merkmale.index.to_series()
        .str.extract(r"^(.+?)[A-Za-z]\d{4}$", expand=False)
        .fillna(FEHLT_WERT)
    )

    for merkmal in produkt_merkmale.columns:
        features[merkmal] = paare["erzeugnis"].map(produkt_merkmale[merkmal])
    for merkmal in komponenten_merkmale.columns:
        features[merkmal] = paare["komponente"].map(komponenten_merkmale[merkmal])

    assert features.notna().all().all(), f"Big1-Merkmale unvollständig für {hg}"

    ergebnisse = paare.copy()
    ergebnisse["p_anwesend"] = np.nan
    ergebnisse["pruefstatus"] = "noch_nicht_geprueft"
    modelle = {}
    spalten_je_fold = {}

    for fold in sorted(paare["fold"].unique()):
        train = paare["fold"].ne(fold)
        pruefen = paare["fold"].eq(fold)
        trainingspaare = paare.loc[train]

        anzahl = (
            trainingspaare.groupby(["komponente", "vorhanden"])["erzeugnis"]
            .nunique()
            .unstack(fill_value=0)
            .reindex(index=X.columns, columns=[0, 1], fill_value=0)
        )

        status = pd.Series("modell_bewertbar", index=anzahl.index)
        status.loc[(anzahl[0] < MIN_PRODUKTE_BIG1) | (anzahl[1] < MIN_PRODUKTE_BIG1)] = "wenig_trainingsbelege"
        status.loc[anzahl[0].eq(0)] = "im_training_immer_anwesend"
        status.loc[anzahl[1].eq(0)] = "im_training_nie_anwesend"

        ergebnisse.loc[pruefen, "pruefstatus"] = paare.loc[pruefen, "komponente"].map(status)

        geeignete = status.index[status.eq("modell_bewertbar")]
        fit_maske = train & paare["komponente"].isin(geeignete)
        score_maske = pruefen & paare["komponente"].isin(geeignete)

        spalten = [
            f"kontext_{i}"
            for i, komponente in enumerate(X.columns)
            if anzahl.loc[komponente, 1] > 0
        ] + BIG1_KATEGORIAL

        if fit_maske.any() and np.unique(y[fit_maske.to_numpy()]).size == 2:
            modell = CatBoostClassifier(
                iterations=300,
                depth=6,
                learning_rate=0.05,
                loss_function="Logloss",
                random_seed=SEED + int(fold),
                thread_count=4,
                verbose=False,
                allow_writing_files=False,
            )
            modell.fit(
                features.loc[fit_maske, spalten],
                y[fit_maske.to_numpy()],
                cat_features=BIG1_KATEGORIAL,
                sample_weight=paare.loc[fit_maske, "produktgewicht"].to_numpy(),
            )
            ergebnisse.loc[score_maske, "p_anwesend"] = modell.predict_proba(
                features.loc[score_maske, spalten]
            )[:, 1]
            modelle[fold] = modell
            spalten_je_fold[fold] = spalten

        auswertung_big1.append({
            "Hauptgruppe": hg,
            "Fold": fold,
            "Trainingspaare": int(fit_maske.sum()),
            "Bewertete Prüfpaare": int(ergebnisse.loc[pruefen, "p_anwesend"].notna().sum()),
            "Separat zu prüfen": int((pruefen & ~score_maske).sum()),
        })

    ergebnisse["auff_big1"] = np.where(
        ergebnisse["vorhanden"].eq(0),
        ergebnisse["p_anwesend"],
        1 - ergebnisse["p_anwesend"],
    )
    ergebnisse["pruefrichtung"] = np.where(
        ergebnisse["vorhanden"].eq(0),
        "möglicherweise fehlend",
        "möglicherweise zusätzlich",
    )
    ergebnisse["hauptgruppe"] = hg

    paare_je_hauptgruppe[hg] = paare
    y_je_hauptgruppe[hg] = y
    features_big1_je_hauptgruppe[hg] = features
    modelle_big1_je_hauptgruppe[hg] = modelle
    spalten_big1_je_hauptgruppe[hg] = spalten_je_fold
    ergebnisse_big1_je_hauptgruppe[hg] = ergebnisse

big1_laufbericht = pd.DataFrame(auswertung_big1)
ergebnisse_big1 = pd.concat(ergebnisse_big1_je_hauptgruppe.values(), ignore_index=True)

display(big1_laufbericht)

In [0]:
STRUKTUR_B3 = [
    "b3_traeger_da",
    "b3_traeger_aufgeloest",
    "b3_traeger_ist_erzeugnis",
    "b3_stelle_anteil",
    "b3_stelle_belegt",
    "b3_stelle_belegt_menge",
    "b3_stelle_alpgr",
    "b3_stelle_ewahr",
    "b3_x_alpgr_anteil",
    "b3_x_ewahr_median",
    "b3_x_traeger_phantom_anteil",
    "b3_x_menge_median",
    "b3_nachbar_anteil_x",
    "b3_nachbar_jaccard",
]


def haeufigster(werte):
    vc = pd.Series(werte).value_counts()
    return vc.index[0] if not vc.empty else None


def logloss_w(p, y, w):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    return np.average(-(y * np.log(p) + (1 - y) * np.log(1 - p)), weights=w)


struktur_je_hauptgruppe = {}
ergebnisse_big3_je_hauptgruppe = {}
modelle_big3_je_hauptgruppe = {}
auswertung_big3 = []

for hg in HAUPTGRUPPEN:
    paare = paare_je_hauptgruppe[hg]
    basis_features = features_big1_je_hauptgruppe[hg]
    basis_modelle = modelle_big1_je_hauptgruppe[hg]
    basis_spalten = spalten_big1_je_hauptgruppe[hg]
    ergebnisse_big1_hg = ergebnisse_big1_je_hauptgruppe[hg]
    y = y_je_hauptgruppe[hg]
    rep_bkey = rep_bkey_je_fassung_je_hg[hg]

    S = q.loc[q["hauptgruppe"].eq(hg)].copy()
    S["bkey"] = S["erzeugnis"] + "|" + S["block"]

    node_komp = S.set_index("node")["komponente"]
    S["eltern_komp"] = S["eltern"].map(node_komp)
    S.loc[S["tiefe_num"].eq(1) | S["eltern_komp"].isna(), "eltern_komp"] = "@ERZEUGNIS"

    S["hochgezogen"] = S["stufe_num"].gt(S["tiefe_num"]).fillna(False).astype(int)
    S["stelle"] = (
        S["stufe_num"].astype("Int64").astype(str)
        + "|"
        + S["slot"].astype(str).str.split(" | ", regex=False).str[-1]
    )
    S["hat_alpgr"] = S["alt_gruppe"].fillna("").astype(str).str.strip().ne("")
    S["ewahr"] = pd.to_numeric(S["alt_wahrsch"], errors="coerce").fillna(-1.0)

    komps = [str(c) for c in X_je_hauptgruppe[hg].columns]
    bloecke = (
        block_fassungen_je_hauptgruppe[hg]
        .drop_duplicates("bkey")
        .sort_values(["erzeugnis", "block"])
        .set_index("bkey")[["erzeugnis", "fassung_id"]]
    )

    P_np = (
        S.loc[S["komponente"].isin(komps)]
        .groupby(["bkey", "komponente"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=bloecke.index, columns=komps, fill_value=0)
        .gt(0)
        .astype(np.int8)
        .to_numpy()
    )

    erz_np = bloecke["erzeugnis"].to_numpy()
    fold_block = bloecke["erzeugnis"].map(fold_je_hauptgruppe[hg]).to_numpy()
    Sk = S.loc[S["komponente"].isin(komps)].copy()
    mats_je_block = S.groupby("bkey")["komponente"].agg(set).to_dict()
    eltern_je_block = S.groupby("bkey")["eltern_komp"].agg(set).to_dict()
    stelle_rows = (
        S.groupby(["bkey", "stelle"])
        .agg(komps=("komponente", list), menge=("menge_num", list), alpgr=("hat_alpgr", list), ewahr=("ewahr", list))
        .to_dict("index")
    )

    Pf = P_np.astype(np.float32)
    schnitt = Pf @ Pf.T
    groesse = Pf.sum(axis=1)
    union = groesse[:, None] + groesse[None, :] - schnitt
    anderes_produkt = erz_np[:, None] != erz_np[None, :]
    k_nb = min(K_NB_B3, max(1, len(bloecke) - 1))

    def struktur_fuer_fold(fold):
        tb = fold_block != fold
        tr = Sk.loc[Sk["bkey"].isin(set(bloecke.index[tb]))]
        st = (
            tr.groupby(["komponente", "stelle"]).size().rename("n").reset_index()
            .sort_values(["komponente", "n"], ascending=[True, False])
            .drop_duplicates("komponente")
            .set_index("komponente")
        )
        st["anteil"] = st["n"] / tr.groupby("komponente").size().reindex(st.index)
        st["traeger"] = tr.groupby("komponente")["eltern_komp"].agg(haeufigster).reindex(st.index)
        xst = tr.groupby("komponente").agg(
            alpgr_anteil=("hat_alpgr", "mean"),
            ewahr_median=("ewahr", "median"),
            phantom_anteil=("hochgezogen", "mean"),
            menge_median=("menge_num", "median"),
        )
        zeilen = []
        for x_i, X in enumerate(komps):
            bekannt = X in st.index
            traeger = st.at[X, "traeger"] if bekannt else None
            stelle = st.at[X, "stelle"] if bekannt else None
            bx = Pf[:, x_i]
            sx = schnitt - np.outer(bx, bx)
            ux = union - np.maximum(bx[:, None], bx[None, :])
            jx = np.where(anderes_produkt & tb[None, :], sx / np.maximum(ux, 1), -1.0)
            top = np.argpartition(-jx, kth=k_nb - 1, axis=1)[:, :k_nb]
            top_j = np.take_along_axis(jx, top, axis=1)
            gueltig = top_j >= 0
            anteil_x = np.where(gueltig, P_np[top, x_i], 0).sum(1) / np.maximum(gueltig.sum(1), 1)
            jac_mean = np.where(gueltig, top_j, 0).sum(1) / np.maximum(gueltig.sum(1), 1)

            for b_i, bkey in enumerate(bloecke.index):
                z = {
                    "bkey": bkey,
                    "komponente": X,
                    "b3_nachbar_anteil_x": anteil_x[b_i],
                    "b3_nachbar_jaccard": jac_mean[b_i],
                }
                if bekannt:
                    ist_erz = traeger == "@ERZEUGNIS"
                    a = stelle_rows.get((bkey, stelle))
                    idx = [i for i, k in enumerate(a["komps"]) if k != X] if a else []
                    z.update({
                        "b3_traeger_ist_erzeugnis": int(ist_erz),
                        "b3_traeger_da": int(ist_erz or traeger in mats_je_block.get(bkey, set())),
                        "b3_traeger_aufgeloest": int(ist_erz or traeger in eltern_je_block.get(bkey, set())),
                        "b3_stelle_anteil": st.at[X, "anteil"],
                        "b3_stelle_belegt": len(idx),
                        "b3_stelle_belegt_menge": max(a["menge"][i] for i in idx) if idx else 0.0,
                        "b3_stelle_alpgr": int(any(a["alpgr"][i] for i in idx)) if idx else 0,
                        "b3_stelle_ewahr": max(a["ewahr"][i] for i in idx) if idx else -1.0,
                        "b3_x_alpgr_anteil": xst.at[X, "alpgr_anteil"],
                        "b3_x_ewahr_median": xst.at[X, "ewahr_median"],
                        "b3_x_traeger_phantom_anteil": xst.at[X, "phantom_anteil"],
                        "b3_x_menge_median": xst.at[X, "menge_median"],
                    })
                zeilen.append(z)
        f = pd.DataFrame(zeilen)
        for c in STRUKTUR_B3:
            if c not in f.columns:
                f[c] = np.nan
        return f.fillna({c: -1.0 for c in STRUKTUR_B3}).set_index(["bkey", "komponente"])

    struktur_je_fold = {}
    paare_bkey = paare["fassung_id"].map(rep_bkey)
    paare_komp = paare["komponente"].astype(str)

    for fold in sorted(paare["fold"].unique()):
        sf = struktur_fuer_fold(fold)
        struktur_je_fold[fold] = sf.reindex(
            pd.MultiIndex.from_arrays([paare_bkey, paare_komp])
        ).set_index(paare.index)
        assert struktur_je_fold[fold][STRUKTUR_B3].notna().all().all(), f"Big3-Struktur fehlt für {hg}, Fold {fold}"

    ergebnisse_b3 = ergebnisse_big1_hg.copy()
    ergebnisse_b3["p_big3"] = np.nan
    modelle_big3 = {}

    for fold in sorted(basis_modelle):
        pruefen = paare["fold"].eq(fold) & ergebnisse_big1_hg["p_anwesend"].notna()
        komponenten = paare.loc[pruefen, "komponente"].unique()
        trainieren = paare["fold"].ne(fold) & paare["komponente"].isin(komponenten)

        X_all = basis_features.join(struktur_je_fold[fold][STRUKTUR_B3])
        spalten = list(basis_spalten[fold]) + STRUKTUR_B3

        parameter = basis_modelle[fold].get_params().copy()
        parameter.pop("cat_features", None)
        modell = CatBoostClassifier(**parameter)
        modell.fit(
            X_all.loc[trainieren, spalten],
            y[trainieren.to_numpy()],
            cat_features=BIG1_KATEGORIAL,
            sample_weight=paare.loc[trainieren, "produktgewicht"].to_numpy(),
        )
        ergebnisse_b3.loc[pruefen, "p_big3"] = modell.predict_proba(
            X_all.loc[pruefen, spalten]
        )[:, 1]
        modelle_big3[fold] = modell

        r = ergebnisse_b3.loc[pruefen]
        y_fold = r["vorhanden"].to_numpy()
        w_fold = r["produktgewicht"].to_numpy()
        auswertung_big3.append({
            "Hauptgruppe": hg,
            "Fold": fold,
            "Paare": len(r),
            "LogLoss Big1": logloss_w(r["p_anwesend"], y_fold, w_fold),
            "LogLoss Big3": logloss_w(r["p_big3"], y_fold, w_fold),
            "Brier Big1": np.average((r["p_anwesend"] - y_fold) ** 2, weights=w_fold),
            "Brier Big3": np.average((r["p_big3"] - y_fold) ** 2, weights=w_fold),
        })

    ergebnisse_b3["auff_big3"] = np.where(
        ergebnisse_b3["vorhanden"].eq(0),
        ergebnisse_b3["p_big3"],
        1 - ergebnisse_b3["p_big3"],
    )
    ergebnisse_b3["hauptgruppe"] = hg

    struktur_je_hauptgruppe[hg] = struktur_je_fold
    ergebnisse_big3_je_hauptgruppe[hg] = ergebnisse_b3
    modelle_big3_je_hauptgruppe[hg] = modelle_big3

if auswertung_big3:
    auswertung_big3 = pd.DataFrame(auswertung_big3)
    display(auswertung_big3.round(5))
else:
    auswertung_big3 = pd.DataFrame()
    print("Keine Big3-Modelle trainiert (Big1 hatte keine geeigneten Modelle).")

if ergebnisse_big3_je_hauptgruppe:
    ergebnisse_big3 = pd.concat(ergebnisse_big3_je_hauptgruppe.values(), ignore_index=True)
else:
    ergebnisse_big3 = pd.DataFrame()

In [0]:
def menge_key_b2(wert):
    return format(float(wert), ".12g")


def auswerten_big2(faelle, wahrscheinlichkeiten, klasseninfo, trainingskomponenten):
    p = np.asarray(wahrscheinlichkeiten, dtype=float)
    assert p.shape == (len(faelle), len(klasseninfo))
    assert np.allclose(p.sum(axis=1), 1)
    q_keys = list(dict.fromkeys(klasseninfo["menge_key"]))
    q_mask = np.array([klasseninfo["menge_key"].eq(q).to_numpy() for q in q_keys])
    pq = p @ q_mask.T
    ergebnisse = []

    for i, (_, fall) in enumerate(faelle.iterrows()):
        ist_q = menge_key_b2(fall["menge_summe"])
        ist_n = int(fall["anzahl_positionen"])
        best_q_index = int(pq[i].argmax())
        best_q_key = q_keys[best_q_index]
        p_best = float(pq[i, best_q_index])
        p_ist = float(pq[i, q_keys.index(ist_q)]) if ist_q in q_keys else np.nan
        if np.isfinite(p_ist) and abs(p_best - p_ist) <= 1e-12:
            best_q_key = ist_q

        gesehen = fall["komponente"] in trainingskomponenten
        grund, kandidat, score = "", False, np.nan
        status = "bewertbar"
        expected_n, p_n_ist, p_n_best = np.nan, np.nan, np.nan

        if ist_q in q_keys:
            mask = klasseninfo["menge_key"].eq(ist_q).to_numpy()
            n_values = sorted(klasseninfo.loc[mask, "anzahl_positionen"].unique())
            pn = np.array([
                p[i, mask & klasseninfo["anzahl_positionen"].eq(v).to_numpy()].sum()
                for v in n_values
            ])
            pn = pn / pn.sum() if pn.sum() > 0 else np.zeros_like(pn)
            expected_n = int(n_values[int(pn.argmax())])
            p_n_best = float(pn.max())
            if ist_n in n_values:
                p_n_ist = float(pn[n_values.index(ist_n)])

        if not gesehen:
            status, grund = "komponente_nicht_im_training", "Komponente im Training unbekannt"
        elif not np.isfinite(p_ist):
            status, grund, kandidat = "menge_nicht_im_training", "Menge im Training unbekannt", True
            score = 1.0
        elif best_q_key != ist_q and p_best > p_ist + 1e-12:
            grund, kandidat, score = "Menge abweichend", True, 1 - p_ist
        elif not np.isfinite(p_n_ist):
            status, kandidat = "positionszahl_bei_menge_unbekannt", True
            grund = "möglicherweise doppelte Position" if ist_n > expected_n else "Positionszahl im Training unbekannt"
            score = 1.0
        elif expected_n != ist_n and p_n_best > p_n_ist + 1e-12:
            grund = "möglicherweise doppelte Position" if ist_n > expected_n else "Positionszahl abweichend"
            kandidat, score = True, 1 - p_n_ist

        ergebnisse.append({
            "beobachtete_menge": float(fall["menge_summe"]),
            "erwartete_menge": float(best_q_key) if gesehen else np.nan,
            "erwartete_positionszahl_bei_ist_menge": expected_n if gesehen else np.nan,
            "p_ist_menge": p_ist if gesehen else np.nan,
            "p_erwartete_menge": p_best if gesehen else np.nan,
            "p_ist_positionszahl_bedingt": p_n_ist if gesehen else np.nan,
            "kandidat_big2": kandidat,
            "grund_big2": grund,
            "auffaelligkeit_big2": score,
            "pruefstatus_big2": status,
        })

    return pd.concat([faelle.reset_index(drop=True), pd.DataFrame(ergebnisse)], axis=1)


def prognose_big2(modell, train_faelle, test_faelle, test_features):
    info = train_faelle.drop_duplicates("zustand_big2").set_index("zustand_big2")
    info = info.loc[np.asarray(modell.classes_).astype(str)]
    return auswerten_big2(
        test_faelle,
        modell.predict_proba(test_features),
        info,
        set(train_faelle["komponente"]),
    )


fold_je_produkt = {
    produkt: fold
    for hg, mapping in fold_je_hauptgruppe.items()
    for produkt, fold in mapping.items()
}

big2_positionen = q.loc[q["tiefe_num"].eq(1)].copy()
assert big2_positionen["erzeugnis"].isin(fold_je_produkt).all()
big2_positionen["fold"] = big2_positionen["erzeugnis"].map(fold_je_produkt)

faelle_b2 = (
    big2_positionen.groupby(["erzeugnis", "block", "komponente"], sort=False)["menge_num"]
    .agg(lambda werte: tuple(sorted(werte)))
    .reset_index(name="mengen_je_position")
)
faelle_b2["anzahl_positionen"] = faelle_b2["mengen_je_position"].map(len)
faelle_b2["menge_summe"] = faelle_b2["mengen_je_position"].map(sum)
faelle_b2["menge_key"] = faelle_b2["menge_summe"].map(menge_key_b2)
faelle_b2["zustand_big2"] = faelle_b2["menge_key"] + "__" + faelle_b2["anzahl_positionen"].astype(str)
faelle_b2["hauptgruppe"] = faelle_b2["erzeugnis"].map(L2.hauptgruppe_von)
faelle_b2["fold"] = faelle_b2["erzeugnis"].map(fold_je_produkt)

# Kontext-Features werden jetzt je Hauptgruppe gebaut (nicht global),
# damit jedes Modell nur Spalten seiner eigenen Gruppe sieht.
blockanzahl_b2 = faelle_b2[["erzeugnis", "block"]].drop_duplicates().groupby("erzeugnis").size()
faelle_b2["blockgewicht"] = 1.0 / faelle_b2["erzeugnis"].map(blockanzahl_b2)

_prod_spannung = q.groupby("erzeugnis")["erzeugnis_txt"].agg(lambda s: eindeutig(s.map(spannung_text)))
_komp_spannung = q.groupby("komponente")["komponente_txt"].agg(lambda s: eindeutig(s.map(spannung_text)))

features_big2_je_hg = {}
modellspalten_big2_je_hg = {}

# Nachbar-Merkmale für Big2 (Jaccard über alle Blöcke, gefiltert nach HG im Training)
f_nb = faelle_b2.copy()
bloecke_nb = f_nb[["erzeugnis", "block", "hauptgruppe", "fold"]].drop_duplicates().reset_index(drop=True)
block_idx_nb = pd.MultiIndex.from_frame(bloecke_nb[["erzeugnis", "block"]])
komp_idx_nb = pd.Index(sorted(f_nb["komponente"].unique()))
zeile_je_fall = block_idx_nb.get_indexer(pd.MultiIndex.from_frame(f_nb[["erzeugnis", "block"]]))
spalte_je_fall_nb = komp_idx_nb.get_indexer(f_nb["komponente"])

B = np.zeros((len(bloecke_nb), len(komp_idx_nb)), dtype=np.float32)
Q = np.full((len(bloecke_nb), len(komp_idx_nb)), np.nan, dtype=np.float32)
B[zeile_je_fall, spalte_je_fall_nb] = 1.0
Q[zeile_je_fall, spalte_je_fall_nb] = f_nb["menge_summe"].to_numpy(dtype=np.float32)

schnitt = B @ B.T
groesse = B.sum(axis=1)
vereinigung = groesse[:, None] + groesse[None, :] - schnitt
jaccard = np.where(vereinigung > 0, schnitt / np.maximum(vereinigung, 1e-9), 0.0)

hg_block = bloecke_nb["hauptgruppe"].to_numpy()
fold_block = bloecke_nb["fold"].to_numpy()
erz_block = bloecke_nb["erzeugnis"].to_numpy()
k_nb = min(K_NACHBARN_BIG2, max(1, len(bloecke_nb) - 1))
nachbar_merkmale_nb = {}

# Kontext-Features je Hauptgruppe bauen
for hg in HAUPTGRUPPEN:
    fb_hg = faelle_b2.loc[faelle_b2["hauptgruppe"].eq(hg)]
    if fb_hg.empty:
        continue
    bi_hg = pd.MultiIndex.from_frame(fb_hg[["erzeugnis", "block"]].drop_duplicates())
    komps_hg = sorted(fb_hg["komponente"].unique())
    pos_m = (
        fb_hg.pivot(index=["erzeugnis", "block"], columns="komponente", values="anzahl_positionen")
        .reindex(index=bi_hg, columns=komps_hg).fillna(0).to_numpy(dtype=np.float32)
    )
    men_m = (
        fb_hg.pivot(index=["erzeugnis", "block"], columns="komponente", values="menge_summe")
        .reindex(index=bi_hg, columns=komps_hg).fillna(0).to_numpy(dtype=np.float32)
    )
    bj = bi_hg.get_indexer(pd.MultiIndex.from_frame(fb_hg[["erzeugnis", "block"]]))
    sj = pd.Index(komps_hg).get_indexer(fb_hg["komponente"])
    zz = np.arange(len(fb_hg))
    kp = pos_m[bj].copy(); km = men_m[bj].copy()
    kp[zz, sj] = 0; km[zz, sj] = 0
    feat = pd.DataFrame(
        np.concatenate([kp, km], axis=1),
        index=fb_hg.index,
        columns=[f"anzahl_kontext_{i}" for i in range(len(komps_hg))]
        + [f"menge_kontext_{i}" for i in range(len(komps_hg))],
    )
    feat["pruefkomponente"] = fb_hg["komponente"].astype(str).to_numpy()
    feat["produkt_spannung"] = fb_hg["erzeugnis"].map(_prod_spannung).fillna(FEHLT_WERT)
    feat["komponente_spannung"] = fb_hg["komponente"].map(_komp_spannung).fillna(FEHLT_WERT)
    feat["komponente_basis"] = (
        fb_hg["komponente"].astype(str)
        .str.extract(r"^(.+?)[A-Za-z]\d{4}$", expand=False)
        .fillna(FEHLT_WERT)
    )
    features_big2_je_hg[hg] = feat
    modellspalten_big2_je_hg[hg] = list(feat.columns)

for fold in sorted(f_nb["fold"].unique()):
    sim = jaccard.copy()
    sim[:, fold_block == fold] = -1.0
    sim[hg_block[:, None] != hg_block[None, :]] = -1.0
    sim[erz_block[:, None] == erz_block[None, :]] = -1.0

    top = np.argpartition(-sim, kth=k_nb - 1, axis=1)[:, :k_nb]
    top_sim = np.take_along_axis(sim, top, axis=1)
    gueltig = top_sim > 0

    nb_zeilen = top[zeile_je_fall]
    nb_gueltig = gueltig[zeile_je_fall]
    nb_menge = Q[nb_zeilen, spalte_je_fall_nb[:, None]]
    nb_menge = np.where(nb_gueltig, nb_menge, np.nan)
    hat = ~np.isnan(nb_menge)

    m = pd.DataFrame(index=f_nb.index)
    _valid_sim = np.where(nb_gueltig, top_sim[zeile_je_fall], 0.0)
    _valid_count = nb_gueltig.sum(axis=1)
    mean_jaccard = np.where(_valid_count > 0, _valid_sim.sum(axis=1) / _valid_count, 0.0)
    m["nb_jaccard_mittel"] = mean_jaccard
    m["nb_anteil_mit_komponente"] = hat.sum(axis=1) / np.maximum(nb_gueltig.sum(axis=1), 1)
    m["nb_menge_median"] = -1.0
    m["nb_menge_min"] = -1.0
    m["nb_menge_max"] = -1.0
    hat_nachbarn = hat.sum(axis=1) > 0
    if hat_nachbarn.any():
        idx = np.where(hat_nachbarn)[0]
        m.loc[f_nb.index[idx], "nb_menge_median"] = np.nanmedian(nb_menge[idx], axis=1)
        m.loc[f_nb.index[idx], "nb_menge_min"] = np.nanmin(nb_menge[idx], axis=1)
        m.loc[f_nb.index[idx], "nb_menge_max"] = np.nanmax(nb_menge[idx], axis=1)
    nachbar_merkmale_nb[fold] = m

NB_MODELLMERKMALE = [
    "nb_jaccard_mittel",
    "nb_anteil_mit_komponente",
    "nb_menge_median",
    "nb_menge_min",
    "nb_menge_max",
]

KATEGORIALE_BIG2 = [
    "pruefkomponente",
    "produkt_spannung",
    "komponente_spannung",
    "komponente_basis",
]

modelle_big2 = {}
ergebnisse_big2_teile = []
auswertung_big2 = []

for hg in HAUPTGRUPPEN:
    if hg not in features_big2_je_hg:
        continue
    feat_hg = features_big2_je_hg[hg]
    mspalten_hg = modellspalten_big2_je_hg[hg] + NB_MODELLMERKMALE
    hg_folds = sorted(faelle_b2.loc[faelle_b2["hauptgruppe"].eq(hg), "fold"].unique())
    for fold in hg_folds:
        in_hg = faelle_b2["hauptgruppe"].eq(hg)
        train = in_hg & faelle_b2["fold"].ne(fold)
        test = in_hg & faelle_b2["fold"].eq(fold)
        if not train.any() or not test.any():
            continue
        train_klassen = faelle_b2.loc[train, "zustand_big2"].nunique()
        if train_klassen < 2:
            continue

        X_fold = feat_hg.join(nachbar_merkmale_nb[fold][NB_MODELLMERKMALE])[mspalten_hg]
        modell = CatBoostClassifier(
            iterations=250,
            depth=6,
            learning_rate=0.05,
            loss_function="MultiClass",
            random_seed=SEED + int(fold),
            thread_count=4,
            verbose=False,
            allow_writing_files=False,
        )
        modell.fit(
            X_fold.loc[train],
            faelle_b2.loc[train, "zustand_big2"].astype(str),
            cat_features=KATEGORIALE_BIG2,
            sample_weight=faelle_b2.loc[train, "blockgewicht"].to_numpy(),
        )

        erg = prognose_big2(modell, faelle_b2.loc[train], faelle_b2.loc[test], X_fold.loc[test])
        erg.index = faelle_b2.index[test]
        erg["hauptgruppe"] = hg
        ergebnisse_big2_teile.append(erg)
        modelle_big2[(hg, fold)] = modell

        auswertung_big2.append({
            "Hauptgruppe": hg,
            "Fold": fold,
            "Prüffälle": int(test.sum()),
            "Bewertbar": int(erg["p_ist_menge"].notna().sum()),
            "Kandidaten": int(erg["kandidat_big2"].sum()),
        })

if ergebnisse_big2_teile:
    ergebnisse_big2 = pd.concat(ergebnisse_big2_teile).sort_index()
    ergebnisse_big2["erzeugnis_txt"] = ergebnisse_big2["erzeugnis"].map(produktnamen)
    ergebnisse_big2["komponente_txt"] = ergebnisse_big2["komponente"].map(komponentennamen)
else:
    ergebnisse_big2 = pd.DataFrame()
    print("Keine Big2-Modelle trainiert.")

if auswertung_big2:
    display(pd.DataFrame(auswertung_big2))
else:
    print("Kein Big2-Laufbericht.")

In [0]:
# Cmd 7 – ERSETZT die komplette bisherige Cmd 7
# Voraussetzungen: ergebnisse_big3_je_hauptgruppe, rep_block_je_fassung_je_hg, produktnamen,
# komponentennamen, ergebnisse_big2, HAUPTGRUPPEN, TOP_PRODUKTE, MAX_LISTENZEILEN (Cmd 2–6)
# Änderung: Produktranking nach -log(1-s) statt -log(s); Listenlimit je Hauptgruppe.

praesenz_teile = []
for hg in HAUPTGRUPPEN:
    t = ergebnisse_big3_je_hauptgruppe[hg].copy()
    t["block"] = t["fassung_id"].map(rep_block_je_fassung_je_hg[hg])
    t["erzeugnis_txt"] = t["erzeugnis"].map(produktnamen)
    t["komponente_txt"] = t["komponente"].map(komponentennamen)
    t["auff_presence"] = np.where(
        t["vorhanden"].eq(0),
        t["auff_big1"] * t["auff_big3"],
        0.5 * (t["auff_big1"] + t["auff_big3"]),
    )
    t["grund_presence"] = np.where(
        t["vorhanden"].eq(0),
        np.where(
            t["auff_big3"].lt(0.2),
            "Big1 meldet fehlend, Big3 schwächt stark ab",
            np.where(
                t["auff_big3"].lt(0.5),
                "Big1 meldet fehlend, Big3 schwächt ab",
                "Big1 meldet fehlend, Big3 bestätigt",
            ),
        ),
        np.where(
            t["auff_big3"].gt(0.8),
            "zusätzliche Komponente stark bestätigt",
            "zusätzliche Komponente auffällig",
        ),
    )
    t = t.loc[t["p_big3"].notna()].copy()
    t = t.loc[
        ((t["vorhanden"].eq(0)) & t["auff_presence"].ge(0.35))
        | ((t["vorhanden"].eq(1)) & t["auff_presence"].ge(0.60))
    ].copy()
    praesenz_teile.append(t[[
        "hauptgruppe",
        "erzeugnis",
        "block",
        "fassung_id",
        "komponente",
        "erzeugnis_txt",
        "komponente_txt",
        "vorhanden",
        "pruefrichtung",
        "auff_big1",
        "auff_big3",
        "auff_presence",
        "grund_presence",
    ]])

if praesenz_teile:
    praesenz_kandidaten = pd.concat(praesenz_teile, ignore_index=True)
else:
    praesenz_kandidaten = pd.DataFrame(columns=[
        "hauptgruppe", "erzeugnis", "block", "fassung_id", "komponente",
        "erzeugnis_txt", "komponente_txt", "vorhanden", "pruefrichtung",
        "auff_big1", "auff_big3", "auff_presence", "grund_presence",
    ])

_big2_cols = ["hauptgruppe", "erzeugnis", "block", "komponente", "erzeugnis_txt",
              "komponente_txt", "beobachtete_menge", "erwartete_menge", "grund_big2", "auffaelligkeit_big2"]
if not ergebnisse_big2.empty and "kandidat_big2" in ergebnisse_big2.columns:
    big2_kandidaten = ergebnisse_big2.loc[
        ergebnisse_big2["kandidat_big2"] & ergebnisse_big2["auffaelligkeit_big2"].notna()
    ][_big2_cols].copy()
else:
    big2_kandidaten = pd.DataFrame(columns=_big2_cols)

anomalien_gesamt = praesenz_kandidaten.merge(
    big2_kandidaten,
    on=["hauptgruppe", "erzeugnis", "block", "komponente", "erzeugnis_txt", "komponente_txt"],
    how="outer",
)

if anomalien_gesamt.empty:
    # Kein Modell konnte Kandidaten finden — leere Ergebnisse mit korrekten Spalten
    _review_cols = ["hauptgruppe", "rang_produkt", "erzeugnis", "erzeugnis_txt", "block",
                    "komponente", "komponente_txt", "anomalie_typ", "beobachtete_menge",
                    "erwartete_menge", "grund", "auff_big1", "auff_big3",
                    "auffaelligkeit_big2", "gesamtauffaelligkeit", "kandidaten", "produktbefund"]
    produkt_ranking = pd.DataFrame(columns=["hauptgruppe", "erzeugnis", "block", "rang_produkt"])
    anomalien_review = pd.DataFrame(columns=_review_cols)
    anzeige_review = pd.DataFrame()
    review_produkte = pd.DataFrame(columns=["hauptgruppe", "erzeugnis"])
    summary_final = pd.DataFrame([{
        "Hauptgruppe": hg,
        "Produkte im Review": 0, "Zeilen im Review": 0,
        "Pr\u00e4senzkandidaten": 0, "Mengenkandidaten": 0,
    } for hg in HAUPTGRUPPEN])
    if HG_UEBERSPRUNGEN:
        _skip_text = ", ".join(f"{k} ({v})" for k, v in HG_UEBERSPRUNGEN.items())
        print(f"\n{'='*90}")
        print(f"  Nicht gepr\u00fcft: {_skip_text} \u2013 zu wenige Produkte f\u00fcr ein Modell")
        print(f"{'='*90}\n")
    print("Keine Anomalie-Kandidaten gefunden \u2013 BOM zu klein f\u00fcr die Modelle.")
    display(summary_final)

anomalien_gesamt["anomalie_typ"] = np.select(
    [
        (anomalien_gesamt["auff_presence"].notna() & anomalien_gesamt["auffaelligkeit_big2"].notna()).to_numpy(),
        anomalien_gesamt["auffaelligkeit_big2"].notna().to_numpy(),
        anomalien_gesamt["auff_presence"].notna().to_numpy(),
    ],
    [
        "Anwesenheit und Menge",
        "Menge",
        "Anwesenheit",
    ],
    default="Unbekannt",
)

anomalien_gesamt["grund"] = anomalien_gesamt["grund_presence"].fillna("")
anomalien_gesamt.loc[
    anomalien_gesamt["grund_big2"].notna() & anomalien_gesamt["grund"].ne(""),
    "grund",
] = anomalien_gesamt["grund"] + "; " + anomalien_gesamt["grund_big2"]
anomalien_gesamt["grund"] = anomalien_gesamt["grund"].mask(
    anomalien_gesamt["grund"].eq(""),
    anomalien_gesamt["grund_big2"],
)

max_score = np.fmax(
    anomalien_gesamt["auff_presence"].fillna(-np.inf),
    anomalien_gesamt["auffaelligkeit_big2"].fillna(-np.inf),
)
boost_score = np.where(
    anomalien_gesamt["auff_presence"].notna() & anomalien_gesamt["auffaelligkeit_big2"].notna(),
    0.15 * np.minimum(
        anomalien_gesamt["auff_presence"].fillna(0),
        anomalien_gesamt["auffaelligkeit_big2"].fillna(0),
    ),
    0.0,
)
anomalien_gesamt["gesamtauffaelligkeit"] = np.clip(max_score + boost_score, 0, 1)

produkt_ranking = (
    anomalien_gesamt.groupby(["hauptgruppe", "erzeugnis", "block", "erzeugnis_txt"])
    .agg(
        kandidaten=("komponente", "size"),
        max_auffaelligkeit=("gesamtauffaelligkeit", "max"),
        summe_evidenz=("gesamtauffaelligkeit", lambda s: float(np.sum(-np.log(1 - np.clip(s, 0, 1 - 1e-4))))),
        hat_praesenz=("auff_presence", lambda s: bool(s.notna().any())),
        hat_menge=("auffaelligkeit_big2", lambda s: bool(s.notna().any())),
    )
    .reset_index()
)
produkt_ranking["produktbefund"] = np.select(
    [
        (produkt_ranking["hat_praesenz"].astype(bool) & produkt_ranking["hat_menge"].astype(bool)).to_numpy(),
        produkt_ranking["hat_menge"].astype(bool).to_numpy(),
        produkt_ranking["hat_praesenz"].astype(bool).to_numpy(),
    ],
    [
        "Anwesenheits- und Mengenbefunde",
        "Mengenbefund",
        "Anwesenheitsbefund",
    ],
    default="Unbekannt",
)
produkt_ranking = produkt_ranking.sort_values(
    ["hauptgruppe", "summe_evidenz", "max_auffaelligkeit"],
    ascending=[True, False, False],
)
produkt_ranking["rang_produkt"] = produkt_ranking.groupby("hauptgruppe").cumcount() + 1

review_produkte = produkt_ranking.loc[produkt_ranking["rang_produkt"].le(TOP_PRODUKTE)].copy()
anomalien_review = anomalien_gesamt.merge(
    review_produkte[[
        "hauptgruppe",
        "erzeugnis",
        "block",
        "rang_produkt",
        "kandidaten",
        "produktbefund",
    ]],
    on=["hauptgruppe", "erzeugnis", "block"],
    how="inner",
)
anomalien_review = anomalien_review.sort_values(
    ["hauptgruppe", "rang_produkt", "gesamtauffaelligkeit"],
    ascending=[True, True, False],
).groupby("hauptgruppe", group_keys=False).head(MAX_LISTENZEILEN)

# Globaler Cap über alle Gruppen hinweg
if len(anomalien_review) > MAX_LISTENZEILEN:
    anomalien_review = anomalien_review.sort_values("gesamtauffaelligkeit", ascending=False).head(MAX_LISTENZEILEN)
    print(f"Globaler Cap: {MAX_LISTENZEILEN} Zeilen (von {len(anomalien_gesamt)} Kandidaten insgesamt)")

anzeige_review = anomalien_review[[
    "hauptgruppe",
    "rang_produkt",
    "erzeugnis",
    "erzeugnis_txt",
    "block",
    "komponente",
    "komponente_txt",
    "anomalie_typ",
    "beobachtete_menge",
    "erwartete_menge",
    "grund",
    "auff_big1",
    "auff_big3",
    "auffaelligkeit_big2",
    "gesamtauffaelligkeit",
    "kandidaten",
    "produktbefund",
]].rename(columns={
    "hauptgruppe": "Hauptgruppe",
    "rang_produkt": "Rang Produkt",
    "erzeugnis": "Identnummer",
    "erzeugnis_txt": "Materialkurztext",
    "block": "Block",
    "komponente": "Identnummer Teil",
    "komponente_txt": "Materialkurztext Teil",
    "anomalie_typ": "Typ",
    "beobachtete_menge": "Menge aktuell",
    "erwartete_menge": "Menge erwartet",
    "grund": "Grund",
    "auff_big1": "Auffälligkeit Big1",
    "auff_big3": "Auffälligkeit Big3",
    "auffaelligkeit_big2": "Auffälligkeit Big2",
    "gesamtauffaelligkeit": "Gesamtauffälligkeit",
    "kandidaten": "Kandidaten im Produkt",
    "produktbefund": "Produktbefund",
})

summary_final = pd.DataFrame([
    {
        "Hauptgruppe": hg,
        "Produkte im Review": int(review_produkte.loc[review_produkte["hauptgruppe"].eq(hg), "erzeugnis"].nunique()),
        "Zeilen im Review": int(anomalien_review.loc[anomalien_review["hauptgruppe"].eq(hg)].shape[0]),
        "Präsenzkandidaten": int(praesenz_kandidaten.loc[praesenz_kandidaten["hauptgruppe"].eq(hg)].shape[0]),
        "Mengenkandidaten": int(big2_kandidaten.loc[big2_kandidaten["hauptgruppe"].eq(hg)].shape[0]),
    }
    for hg in HAUPTGRUPPEN
])

# Übersprungene Gruppen deutlich anzeigen
if HG_UEBERSPRUNGEN:
    _skip_text = ", ".join(f"{k} ({v})" for k, v in HG_UEBERSPRUNGEN.items())
    print(f"\n{'='*90}")
    print(f"  Nicht gepr\u00fcft: {_skip_text} \u2013 zu wenige Produkte f\u00fcr ein Modell")
    print(f"{'='*90}\n")

display(summary_final)
if not anzeige_review.empty:
    display(anzeige_review.round(4))

In [0]:
# Cmd 8 – Fachbereichs-Erklärung (universell, Version 3)
# Voraussetzungen: q, HAUPTGRUPPEN, K_NB_B3, K_NACHBARN_BIG2, STRUKTUR_B3, BIG1_KATEGORIAL,
# X_je_hauptgruppe, block_fassungen_je_hauptgruppe, fold_je_hauptgruppe, fold_je_produkt,
# paare_je_hauptgruppe, features_big1_je_hauptgruppe, modelle_big3_je_hauptgruppe,
# spalten_big1_je_hauptgruppe, struktur_je_hauptgruppe, anomalien_review, faelle_b2,
# produktnamen, komponentennamen

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

MAX_FAELLE_JE_HG = 30

def _run_shap():
    """Erzeugt ergebnis-DF. Gibt bei leerer anomalien_review sofort leeren DF zurück."""
    if anomalien_review.empty:
        return pd.DataFrame()

    from collections import Counter
    from catboost import Pool

    MAX_FAELLE_JE_HG = 30          # Anzeige: Fälle je Hauptgruppe (Berechnung läuft für alle)
    TOP_VERGLEICH_TEXT = 5         # Vergleichsprodukte im Beleg

    produktname = {k: str(v).strip() for k, v in produktnamen.to_dict().items()}
    teilname = {k: str(v).strip() for k, v in komponentennamen.to_dict().items()}


    def zahl(v):
        return "–" if v is None or pd.isna(v) else format(float(v), "g")


    def teileart(komp):
        m = re.match(r"[A-Za-zÄÖÜäöüß]+", teilname.get(komp, ""))
        return m.group(0).upper() if m else ""


    def anzahl_text(n, k):
        return "keines" if n == 0 else ("alle" if n == k else f"{n}")


    def zaehler_text(counter, max_n=3):
        return "; ".join(f"{teilname.get(k, k)} ({n}×)" for k, n in counter.most_common(max_n)) or "–"


    # ── Stammdaten je Block ─────────────────────────────────────────────────────────────────
    S_all = q.copy()
    S_all["bkey"] = S_all["erzeugnis"] + "|" + S_all["block"]
    teile_je_block = S_all.groupby("bkey")["komponente"].agg(set).to_dict()
    menge_je_block_teil = S_all.groupby(["bkey", "komponente"])["menge_num"].sum()


    def menge_von(bkey, komp):
        return menge_je_block_teil.get((bkey, komp), np.nan)


    def vergleich_auswerten(eigen_bkey, vergleich_bkeys, X):
        """Was haben die Vergleichsprodukte, was dieses Produkt nicht hat – und umgekehrt."""
        eigen = teile_je_block.get(eigen_bkey, set())
        nur_dort, nur_hier, je_vergleich = Counter(), Counter(), []
        for vb in vergleich_bkeys:
            andere = teile_je_block.get(vb, set())
            d = sorted(andere - eigen - {X})
            h = sorted(eigen - andere - {X})
            nur_dort.update(d)
            nur_hier.update(h)
            je_vergleich.append((d, h))
        return nur_dort, nur_hier, je_vergleich


    def variante_text(X, fehlt, nur_dort, nur_hier):
        """Gleiche Teileart (erstes Wort im Kurztext), andere Ausführung."""
        art = teileart(X)
        if not art:
            return ""
        quelle = nur_hier if fehlt else nur_dort
        kandidaten = [k for k, _ in quelle.most_common() if teileart(k) == art]
        if not kandidaten:
            return ""
        wer = "Dieses Produkt hat stattdessen" if fehlt else "Vergleichsprodukte haben stattdessen"
        return f"{wer} {teilname.get(kandidaten[0], kandidaten[0])} – gleiche Teileart, mögliche Variante"


    def beleg_klasse(anteil):
        if anteil is None or not np.isfinite(anteil):
            return "kein Beleg"
        return "stark" if anteil >= 0.8 else "mittel" if anteil >= 0.5 else "schwach" if anteil > 0 else "kein Beleg"


    SHAP_GRUPPEN = {
        "Vergleichsprodukte": lambda f: f.startswith("b3_nachbar_"),
        "Einbaustelle/Baugruppe": lambda f: f.startswith("b3_stelle_") or f.startswith("b3_traeger_"),
        "Produkttyp/Spannung": lambda f: f in ("kleinere_gruppe", "produkt_spannung"),
        "Eigenschaften des Teils": lambda f: f in ("pruefkomponente", "komponente_spannung", "komponente_basis") or f.startswith("b3_x_"),
        "Andere Teile im Produkt": lambda f: f.startswith("kontext_"),
    }


    def shap_gruppen(spalten, sv, eigene_spalte):
        beitrag = {g: 0.0 for g in SHAP_GRUPPEN}
        for f, s in zip(spalten, sv):
            if f == eigene_spalte:          # maskierte eigene Spalte nicht als "anderes Teil" zählen
                continue
            for g, test in SHAP_GRUPPEN.items():
                if test(f):
                    beitrag[g] += abs(float(s))
                    break
        summe = sum(beitrag.values()) or 1.0
        reihen = sorted(beitrag.items(), key=lambda kv: -kv[1])
        text = ", ".join(f"{g} {b / summe:.0%}" for g, b in reihen if b / summe >= 0.05)
        return reihen[0][0], text


    faelle = []

    # ══════════════════════════ Anwesenheit (Big1/Big3) ════════════════════════════════════
    for hg in HAUPTGRUPPEN:
        rev = anomalien_review.loc[
            anomalien_review["hauptgruppe"].eq(hg) & anomalien_review["auff_presence"].notna()
        ].copy()
        if rev.empty:
            continue
        paare = paare_je_hauptgruppe[hg]
        bf = features_big1_je_hauptgruppe[hg]
        sp_fold = spalten_big1_je_hauptgruppe[hg]
        strk = struktur_je_hauptgruppe[hg]
        komps = [str(c) for c in X_je_hauptgruppe[hg].columns]
        komp_pos = {k: i for i, k in enumerate(komps)}

        S = S_all.loc[S_all["hauptgruppe"].eq(hg)]
        bloecke = (
            block_fassungen_je_hauptgruppe[hg].drop_duplicates("bkey")
            .sort_values(["erzeugnis", "block"]).set_index("bkey")[["erzeugnis", "fassung_id"]]
        )
        bpos = {b: i for i, b in enumerate(bloecke.index)}
        P_np = (
            S.loc[S["komponente"].isin(komps)].groupby(["bkey", "komponente"]).size().unstack(fill_value=0)
            .reindex(index=bloecke.index, columns=komps, fill_value=0).gt(0).astype(np.int8).to_numpy()
        )
        Pf = P_np.astype(np.float32)
        schnitt = Pf @ Pf.T
        groesse = Pf.sum(axis=1)
        union = groesse[:, None] + groesse[None, :] - schnitt
        erz_np = bloecke["erzeugnis"].to_numpy()
        fold_block = bloecke["erzeugnis"].map(fold_je_hauptgruppe[hg]).to_numpy()
        k_nb = min(K_NB_B3, max(1, len(bloecke) - 1))

        idx_map = dict(zip(zip(paare["fassung_id"], paare["komponente"]), paare.index))
        rev["_fold"] = rev["erzeugnis"].map(fold_je_hauptgruppe[hg])

        for fold, mdl in modelle_big3_je_hauptgruppe.get(hg, {}).items():
            batch = rev.loc[rev["_fold"].eq(fold)]
            if batch.empty:
                continue
            tb = fold_block != fold
            spalten = list(sp_fold[fold]) + STRUKTUR_B3
            X_all = bf.join(strk[fold][STRUKTUR_B3])
            rows = [idx_map.get((int(r["fassung_id"]), r["komponente"])) for _, r in batch.iterrows()]
            gueltig = [i for i, r in enumerate(rows) if r is not None]
            if not gueltig:
                continue
            X_b = X_all.loc[[rows[i] for i in gueltig], spalten]
            cat_pos = [spalten.index(c) for c in BIG1_KATEGORIAL if c in spalten]
            shap_m = mdl.get_feature_importance(Pool(X_b, cat_features=cat_pos), type="ShapValues")

            for j, i in enumerate(gueltig):
                r = batch.iloc[i]
                X = r["komponente"]
                bkey = f'{r["erzeugnis"]}|{r["block"]}'
                if bkey not in bpos or X not in komp_pos:
                    continue
                b_i, x_i = bpos[bkey], komp_pos[X]
                bx = Pf[:, x_i]
                sx = schnitt[b_i] - bx[b_i] * bx
                ux = union[b_i] - np.maximum(bx[b_i], bx)
                jx = np.where((erz_np != erz_np[b_i]) & tb, sx / np.maximum(ux, 1), -1.0)
                top = [t for t in np.argsort(-jx)[:k_nb] if jx[t] >= 0]
                vbkeys = [bloecke.index[t] for t in top]

                fehlt = int(r["vorhanden"]) == 0
                n_mit = int(sum(P_np[t, x_i] for t in top))
                k = len(top)
                mengen_vgl = [menge_von(vb, X) for vb in vbkeys if P_np[bpos[vb], x_i]]
                menge_typisch = pd.Series(mengen_vgl).mode().iloc[0] if mengen_vgl else np.nan
                eigene_menge = menge_von(bkey, X)

                eigene_spalte = f"kontext_{x_i}"
                hauptgrund, _ = shap_gruppen(spalten, shap_m[j, :-1], eigene_spalte)
                nur_dort, nur_hier, _ = vergleich_auswerten(bkey, vbkeys, X)
                anteil = (n_mit / k if fehlt else 1 - n_mit / k) if k else np.nan

                if fehlt:
                    befund = f"Teil fehlt (erwartet {zahl(menge_typisch)}×)" if n_mit else "Teil fehlt"
                    beleg = f"{anzahl_text(n_mit, k)} der {k} ähnlichsten Produkte {'hat' if n_mit == 1 else 'haben'} das Teil" + (f", meist {zahl(menge_typisch)}×" if n_mit else "")
                else:
                    befund = f"Teil zusätzlich ({zahl(eigene_menge)}× enthalten)"
                    beleg = f"{anzahl_text(n_mit, k)} der {k} ähnlichsten Produkte {'hat' if n_mit == 1 else 'haben'} das Teil"

                hinweise = []
                v = variante_text(X, fehlt, nur_dort, nur_hier)
                if v:
                    hinweise.append(v)
                if k and beleg_klasse(anteil) in ("schwach", "kein Beleg"):
                    if hauptgrund in ("Produkttyp/Spannung", "Eigenschaften des Teils"):
                        hinweise.append("Meldung beruht auf Spannung/Teiletyp, nicht auf den Vergleichsprodukten")
                    else:
                        hinweise.append("Vergleichsprodukte stützen die Meldung kaum")
                if k < 3:
                    hinweise.append("Zu wenige Vergleichsprodukte")

                vergleich_kurz = "; ".join(
                    f"{produktname.get(erz_np[t], erz_np[t])} ({'hat es ' + zahl(menge_von(bloecke.index[t], X)) + '×' if P_np[t, x_i] else 'hat es nicht'})"
                    for t in top[:TOP_VERGLEICH_TEXT]
                )

                faelle.append({
                    "hauptgruppe": hg, "rang_produkt": r["rang_produkt"],
                    "erzeugnis": r["erzeugnis"], "block": r["block"], "komponente": X,
                    "befund": befund, "beleg": beleg, "vergleich_kurz": vergleich_kurz,
                    "hinweis": " | ".join(hinweise), "auffaelligkeit": r["gesamtauffaelligkeit"],
                })

    # ══════════════════════════ Menge (Big2) ═══════════════════════════════════════════════
    rev2 = anomalien_review.loc[anomalien_review["auffaelligkeit_big2"].notna()].copy()
    if not rev2.empty:
        bl2 = faelle_b2[["erzeugnis", "block", "hauptgruppe", "fold"]].drop_duplicates().reset_index(drop=True)
        bl2_idx = pd.MultiIndex.from_frame(bl2[["erzeugnis", "block"]])
        kidx = pd.Index(sorted(faelle_b2["komponente"].unique()))
        zi = bl2_idx.get_indexer(pd.MultiIndex.from_frame(faelle_b2[["erzeugnis", "block"]]))
        si = kidx.get_indexer(faelle_b2["komponente"])
        B2 = np.zeros((len(bl2), len(kidx)), dtype=np.float32)
        Q2 = np.full((len(bl2), len(kidx)), np.nan, dtype=np.float32)
        B2[zi, si] = 1.0
        Q2[zi, si] = faelle_b2["menge_summe"].to_numpy(dtype=np.float32)
        sch2 = B2 @ B2.T
        gr2 = B2.sum(axis=1)
        ver2 = gr2[:, None] + gr2[None, :] - sch2
        jac2 = np.where(ver2 > 0, sch2 / np.maximum(ver2, 1e-9), 0.0)
        hg2, fo2, er2 = bl2["hauptgruppe"].to_numpy(), bl2["fold"].to_numpy(), bl2["erzeugnis"].to_numpy()
        bk2 = (bl2["erzeugnis"] + "|" + bl2["block"]).to_numpy()
        k2 = min(K_NACHBARN_BIG2, max(1, len(bl2) - 1))

        for _, r in rev2.iterrows():
            key = (r["erzeugnis"], r["block"])
            X = r["komponente"]
            if key not in bl2_idx or X not in kidx:
                continue
            b_i, x_i = bl2_idx.get_loc(key), kidx.get_loc(X)
            sim = jac2[b_i].copy()
            sim[(fo2 == fold_je_produkt[r["erzeugnis"]]) | (hg2 != r["hauptgruppe"]) | (er2 == r["erzeugnis"])] = -1.0
            top = [t for t in np.argsort(-sim)[:k2] if sim[t] > 0]
            k = len(top)
            mit = [t for t in top if not np.isnan(Q2[t, x_i])]
            mengen_vgl = [float(Q2[t, x_i]) for t in mit]
            beob, erw = r["beobachtete_menge"], r["erwartete_menge"]
            gleich_beob = sum(abs(m - beob) < 1e-9 for m in mengen_vgl)
            gleich_erw = sum(pd.notna(erw) and abs(m - erw) < 1e-9 for m in mengen_vgl)
            menge_vc = pd.Series(mengen_vgl).map(zahl).value_counts()
            mengen_txt = ", ".join(f"{n}× Menge {m}" for m, n in menge_vc.items())

            befund = f"Menge {zahl(beob)}× statt {zahl(erw)}×"
            beleg = (f"{len(mit)} der {k} ähnlichsten Produkte haben das Teil ({mengen_txt})" if mit
                     else f"keines der {k} ähnlichsten Produkte hat das Teil")
            anteil = gleich_erw / len(mit) if mit else np.nan

            hinweise = []
            if mit and gleich_beob >= max(gleich_erw, 1):
                hinweise.append(f"{gleich_beob} von {len(mit)} Vergleichsprodukten haben dieselbe Menge wie dieses Produkt – wahrscheinlich kein Fehler")
            if not mit:
                hinweise.append("Erwartete Menge ist durch kein Vergleichsprodukt belegt")
            elif gleich_erw == 0:
                hinweise.append("Kein Vergleichsprodukt hat die erwartete Menge")

            vergleich_kurz = "; ".join(
                f"{produktname.get(er2[t], er2[t])} ({'hat es ' + zahl(Q2[t, x_i]) + '×' if not np.isnan(Q2[t, x_i]) else 'hat es nicht'})"
                for t in top[:TOP_VERGLEICH_TEXT]
            )
            faelle.append({
                "hauptgruppe": r["hauptgruppe"], "rang_produkt": r["rang_produkt"],
                "erzeugnis": r["erzeugnis"], "block": r["block"], "komponente": X,
                "befund": befund, "beleg": beleg, "vergleich_kurz": vergleich_kurz,
                "hinweis": " | ".join(hinweise), "auffaelligkeit": r["gesamtauffaelligkeit"],
            })

    # ══════════════════════════ Ein Fall = Produkt + Block + Teil ══════════════════════════
    roh_faelle = pd.DataFrame(faelle)
    SCHLUESSEL = ["hauptgruppe", "erzeugnis", "block", "komponente"]


    def verbinden(s):
        werte = [str(v) for v in s if pd.notna(v) and str(v) != ""]
        return " || ".join(dict.fromkeys(werte))


    def erster_wert(s):
        s = s.dropna()
        return s.iloc[0] if not s.empty else np.nan


    alle_faelle = (
        roh_faelle.groupby(SCHLUESSEL, sort=False)
        .agg(
            rang_produkt=("rang_produkt", "min"),
            befund=("befund", lambda x: " + ".join(dict.fromkeys(str(v) for v in x if pd.notna(v)))),
            beleg=("beleg", verbinden),
            hinweis=("hinweis", verbinden),
            vergleich_kurz=("vergleich_kurz", "first"),
            auffaelligkeit=("auffaelligkeit", "max"),
        )
        .reset_index()
        .sort_values(["hauptgruppe", "rang_produkt", "auffaelligkeit"], ascending=[True, True, False])
    )


    def beleg_gesamt(z):
        teile = [z["hinweis"], z["beleg"]]
        return ". ".join(t.replace(" || ", ". ") for t in teile if t) + "."


    ergebnis = pd.DataFrame({
        "Konfidenz": alle_faelle["auffaelligkeit"].round(3),
        "Produktname": alle_faelle["erzeugnis"].map(produktname),
        "Produktnummer": alle_faelle["erzeugnis"],
        "Teilename": alle_faelle["komponente"].map(teilname),
        "Teilenummer": alle_faelle["komponente"],
        "Befund": alle_faelle["befund"],
        "Beleg": alle_faelle.apply(beleg_gesamt, axis=1),
        "Verglichen mit": alle_faelle["vergleich_kurz"].str.replace("; ", "  |  ", regex=False),
        "_hg": alle_faelle["hauptgruppe"],
    })

    return ergebnis

ergebnis = _run_shap()

print(f"\n{'='*90}")
print(f"  Ausgewertete Stückliste: {BOM_DATEI}")
print(f"{'='*90}")

# Übersprungene Gruppen deutlich anzeigen
if HG_UEBERSPRUNGEN:
    _skip_text = ", ".join(f"{k} ({v})" for k, v in HG_UEBERSPRUNGEN.items())
    print(f"{'='*90}")
    print(f"  Nicht geprüft: {_skip_text} – zu wenige Produkte für ein Modell")
    print(f"{'='*90}")

print(f"\nFälle gesamt: {len(ergebnis)} | angezeigt: bis {MAX_FAELLE_JE_HG} je Hauptgruppe")
if not ergebnis.empty:
    display(ergebnis.groupby("_hg", group_keys=False).head(MAX_FAELLE_JE_HG).drop(columns="_hg"))


In [0]:
# Cmd 9 – Abby-Prüfauftrag
# Voraussetzungen: ergebnis, anomalien_review, BOM_DATEI (Cmd 2–8)

import http.client, json, ssl, os, re
import requests as _req

# ══ Konfiguration ══════════════════════════════════════════════════════════════════
ABBY_AGENT_ID = "b748160a-058f-47e1-a82a-78392c0ebdd9"
ABBY_API_KEY  = "PLACEHOLDER(wurde schon verwendet aber ist im donwload nicht sichtbar)"  # <-- Key eintragen
# ════════════════════════════════════════════════════════════════════════════════

# Resume-Logik: vorhandene Befunde wiederverwenden
try:
    _vorhandene_ids = {b.get("befund_id") for b in alle_befunde if isinstance(b, dict) and "befund_id" in b}
    print(f"Resume: {len(_vorhandene_ids)} Befunde aus vorherigem Lauf vorhanden")
except (NameError, TypeError):
    alle_befunde = []
    _vorhandene_ids = set()

if ergebnis.empty:
    print("⚠  Keine Kandidaten – Abby-Aufruf übersprungen.")
    abby_ergebnisse = {"eingabedatei": BOM_DATEI, "n_kandidaten": 0,
                       "gesendete_daten": [], "antwort_roh": None, "meta": {}}
else:
    # ── Sendetabelle: ergebnis + Befund-ID + Blockkennung ──────────────────────
    _tab = ergebnis.reset_index(drop=True).copy()
    _tab["_row"] = range(len(_tab))
    _tab.rename(columns={"_hg": "Hauptgruppe"}, inplace=True)
    try:
        _bl = (anomalien_review[["hauptgruppe","erzeugnis","block","komponente"]]
               .drop_duplicates()
               .rename(columns={"hauptgruppe":"Hauptgruppe","erzeugnis":"Produktnummer","komponente":"Teilenummer"}))
        _tab = _tab.merge(_bl, on=["Hauptgruppe","Produktnummer","Teilenummer"], how="left")
        _tab = _tab.drop_duplicates(subset=["_row"], keep="first")
    except Exception:
        pass
    _tab.drop(columns=["_row"], errors="ignore", inplace=True)
    _tab.insert(0, "Befund-ID", [f"BF-{i+1:03d}" for i in range(len(_tab))])

    _cols = [c for c in ["Befund-ID","Hauptgruppe","block","Konfidenz","Produktname",
             "Produktnummer","Teilename","Teilenummer","Befund","Beleg","Verglichen mit"]
             if c in _tab.columns]
    sende_df = _tab[_cols].copy()
    n_kandidaten = len(sende_df)

    # ── Prüfauftrag (Batched) ──────────────────────────────────────────────────────
    BATCH_GROESSE = 40
    _headers = {"X-ABBY-API-Key": ABBY_API_KEY, "Content-Type": "application/json"}

    # ── API-Verbindung (http.client mit HTTP/1.1 ALPN fuer ABB-WAF) ───────────
    _ca = os.environ.get("REQUESTS_CA_BUNDLE") or os.environ.get("CURL_CA_BUNDLE") or _req.certs.where()
    _ca_cfg = {"capath": _ca} if os.path.isdir(_ca) else {"cafile": _ca}
    _ctx = ssl.create_default_context(**_ca_cfg)
    _ctx.set_alpn_protocols(["http/1.1"])

    n_batches = -(-n_kandidaten // BATCH_GROESSE)
    print(f"Eingabedatei: {BOM_DATEI}")
    print(f"Kandidaten: {n_kandidaten}")
    print(f"Batches: {n_batches} (je max. {BATCH_GROESSE})")
    print(f"Sende Call an Agent ...\n")

    import time as _time
    _t0 = _time.perf_counter()

    if not _vorhandene_ids:
        alle_befunde = []
    abby_antwort_roh = None
    abby_meta = {}
    _fehler = []

    for _bi in range(n_batches):
        _start = _bi * BATCH_GROESSE
        _ende = min(_start + BATCH_GROESSE, n_kandidaten)
        _batch = sende_df.iloc[_start:_ende]

        # Skip wenn alle IDs dieses Batches schon vorhanden
        _batch_ids = {f"BF-{i+1:03d}" for i in range(_start, _ende)}
        if _batch_ids.issubset(_vorhandene_ids):
            print(f"  Batch {_bi+1}/{n_batches} (BF-{_start+1:03d} bis BF-{_ende:03d}) ... uebersprungen (bereits vorhanden)")
            continue

        _auftrag = (
            "Pruefe jeden uebermittelten Kandidaten nach deinen Agent-Instructions. "
            "Gib die Befund-ID unveraendert zurueck. "
            "Ordne alle Kandidaten nach TRUE POSITIVE -> VERDACHT -> FALSE POSITIVE "
            "und begruende jede Einstufung. "
            "Ungeklaerte Faelle bleiben VERDACHT. Kennzeichne fehlende Informationen. "
            "Erfinde keine familienspezifischen Regeln. "
            "Die uebermittelte Konfidenz ist eine Modell-Rankinggroesse. "
            f"Die Daten gehoeren zur Eingabedatei: {BOM_DATEI}. "
            f"(Batch {_bi+1}/{n_batches})\n\n"
            + _batch.to_json(orient="records", force_ascii=False, indent=2)
        )

        _payload = json.dumps({"agent_id": ABBY_AGENT_ID, "input": _auftrag},
                              ensure_ascii=False).encode("utf-8")

        for _versuch in range(2):
            if _versuch > 0:
                print(f"    Retry in 15s ...")
                _time.sleep(15)
            print(f"  Batch {_bi+1}/{n_batches} (BF-{_start+1:03d} bis BF-{_ende:03d}) ...", end=" ", flush=True)
            try:
                _conn = http.client.HTTPSConnection("api.abby.abb.com", timeout=600, context=_ctx)
                _conn.connect()
                _conn.request("POST", "/api/v1/developers/agent_chat?streaming=false",
                              body=_payload, headers=_headers)
                _resp = _conn.getresponse()
                _body = json.loads(_resp.read().decode("utf-8"))
                _conn.close()

                if _resp.status != 200:
                    print(f"x HTTP {_resp.status}")
                    if _resp.status == 500 and _versuch == 0:
                        continue
                    _fehler.append(f"Batch {_bi+1}: HTTP {_resp.status}")
                    break

                _antwort = _body.get("output", {}).get("content", "")
                _modell = _body.get("model", "?")
                _t_batch = _time.perf_counter() - _t0
                print(f"ok {_modell} ({_t_batch:.0f}s gesamt)", flush=True)

                if not abby_meta:
                    abby_antwort_roh = _body
                    abby_meta = {
                        "status_code": _resp.status,
                        "model": _modell,
                        "conversation_id": _body.get("conversation_id"),
                        "tools_called": _body.get("output", {}).get("tools_called"),
                    }

                # Befunde parsen
                _bf = None
                try:
                    _p = json.loads(_antwort)
                    if isinstance(_p, dict) and "befunde" in _p:
                        _bf = _p["befunde"]
                    elif isinstance(_p, list):
                        _bf = _p
                except (json.JSONDecodeError, TypeError):
                    _m = re.search(r'\{\s*"befunde"\s*:\s*\[.*\]\s*\}', _antwort, re.DOTALL)
                    if _m:
                        try:
                            _bf = json.loads(_m.group())["befunde"]
                        except Exception:
                            pass

                if _bf:
                    alle_befunde.extend(_bf)
                    print(f"    -> {len(_bf)} Befunde geparst")
                else:
                    _fehler.append(f"Batch {_bi+1}: Antwort nicht als JSON parsbar")
                    print(f"    ! Rohtext (nicht parsbar): {_antwort[:200]}")
                break

            except Exception as _e:
                print(f"x {type(_e).__name__}: {_e}")
                _fehler.append(f"Batch {_bi+1}: {type(_e).__name__}: {_e}")
                break

    _dauer = _time.perf_counter() - _t0

    if _fehler:
        print(f"\n! Fehler in {len(_fehler)} Batch(es):")
        for f in _fehler:
            print(f"  - {f}")

    if alle_befunde:
        df_agent = pd.DataFrame(alle_befunde)

        # Join mit Sendetabelle ueber Befund-ID
        _jcol = "befund_id" if "befund_id" in df_agent.columns else None
        if _jcol:
            _merged = sende_df.merge(
                df_agent[[_jcol, "status", "begruendung"]].rename(columns={_jcol: "Befund-ID"}),
                on="Befund-ID", how="left",
            )
        else:
            _merged = sende_df.merge(
                df_agent[["produktnummer", "teilenummer", "status", "begruendung"]].rename(
                    columns={"produktnummer": "Produktnummer", "teilenummer": "Teilenummer"}),
                on=["Produktnummer", "Teilenummer"], how="left",
            )

        _final_cols = ["status", "Befund-ID", "Produktname", "Produktnummer", "Teilename",
                       "Teilenummer", "Befund", "begruendung"]
        if "Verglichen mit" in _merged.columns:
            _final_cols.append("Verglichen mit")
        _final = _merged[_final_cols].copy()
        _final.rename(columns={"status": "Agent Urteil", "Befund-ID": "Befund Nummer", "begruendung": "Begr\u00fcndung"}, inplace=True)

        _so = {"True Positive": 0, "Verdacht": 1, "False Positive": 2}
        _final["_s"] = _final["Agent Urteil"].str.strip().str.title().map(_so).fillna(1)
        _final = _final.sort_values("_s").drop(columns="_s").reset_index(drop=True)

        print(f"\nAgentenbewertung: {len(alle_befunde)} Befunde ({_dauer:.0f}s)")
        _vc = _final["Agent Urteil"].value_counts()
        for s in ["True Positive", "Verdacht"]:
            if s in _vc.index:
                print(f"  {s}: {_vc[s]}")
        display(_final)
    else:
        print("\n! Keine Befunde erhalten.")

    # ── Ergebnisse intern speichern ─────────────────────────────────────────────────
    abby_ergebnisse = {
        "eingabedatei":    BOM_DATEI,
        "n_kandidaten":    n_kandidaten,
        "gesendete_daten": sende_df.to_dict(orient="records"),
        "antwort_text":    json.dumps(alle_befunde, ensure_ascii=False) if alle_befunde else None,
        "antwort_roh":     abby_antwort_roh,
        "meta":            abby_meta,
    }